In [1]:
!pip install -q transformers sentencepiece pillow google-genai pydantic

# Model Donut


In [2]:
import re, time, torch
from PIL import Image
from transformers import DonutProcessor, VisionEncoderDecoderModel

MODEL_ID = "naver-clova-ix/donut-base-finetuned-cord-v2"
processor = DonutProcessor.from_pretrained(MODEL_ID)
donut = VisionEncoderDecoderModel.from_pretrained(MODEL_ID)

device = "cuda" if torch.cuda.is_available() else "cpu"
donut.to(device)
print("Donut siap di:", device)

Loading weights:   0%|          | 0/484 [00:00<?, ?it/s]

Donut siap di: cuda


In [3]:
def read_with_donut(image_path):
    image = Image.open(image_path).convert("RGB")
    pixel_values = processor(image, return_tensors="pt").pixel_values

    prompt_ids = processor.tokenizer(
        "<s_cord-v2>", add_special_tokens=False, return_tensors="pt"
    ).input_ids

    outputs = donut.generate(
        pixel_values.to(device),
        decoder_input_ids=prompt_ids.to(device),
        max_length=donut.decoder.config.max_position_embeddings,
        pad_token_id=processor.tokenizer.pad_token_id,
        eos_token_id=processor.tokenizer.eos_token_id,
        use_cache=True,
        bad_words_ids=[[processor.tokenizer.unk_token_id]],
        return_dict_in_generate=True,
    )

    text = processor.batch_decode(outputs.sequences)[0]
    text = text.replace(processor.tokenizer.eos_token, "")
    text = text.replace(processor.tokenizer.pad_token, "")
    text = re.sub(r"<.*?>", "", text, count=1).strip()
    return processor.token2json(text)

In [4]:
for path in ["/content/nota1.jpg", "/content/nota2.jpg"]:
    start = time.time()
    hasil = read_with_donut(path)
    print("===", path, f"({time.time() - start:.1f} detik)")
    print(hasil)
    print()

=== /content/nota1.jpg (1.7 detik)
{'menu': [{'nm': 'WARUNG HAGIA', 'cnt': {'unitprice': '08577758472'}, 'price': 'Barat', 'sub': {'nm': ': CS/01/26092/0023'}}, {'nm': 'Ice Matcha', 'cnt': '1', 'price': '28,000', 'sub': {'nm': 'Ricebowl Ayam Teriyaki', 'cnt': '1', 'price': '35,000'}}], 'sub_total': {'subtotal_price': '63,000', 'tax_price': '6,300', 'etc': '200'}, 'total_price': '69,500', 'creditcardprice': '69,500', 'subtotal_price': '69,500', 'tax_price': '69,500', 'nm': 'Tarbayar', 'cashprice': 'Salsa Anisa Bila'}

=== /content/nota2.jpg (1.8 detik)
{'menu': [{'nm': 'Delivered at: Zaharwa Rizzani'}, {'nm': 'Kamar No 404 [JL KH. R. Abdullah Bin Nuh.', 'price': '08:00'}, {'nm': 'Status Order: Sedang Diproses', 'price': '08:20'}, {'nm': 'LALADON, CIOMAS', 'num': 'JL SINDANG BARANG RT.05', 'price': {'num': 'Aque', 'unitprice': '20,300', 'cnt': '1', 'nm': 'Laurier Relax Night Pembalut'}}, {'nm': 'Wanita Wing Malam 35 cm 12 pcs', 'unitprice': '10,200', 'cnt': '1', 'price': '10,200'}, {'cnt

# Model gemini-3.8-flash

In [5]:
import time
from PIL import Image
from pydantic import BaseModel
from google import genai
from google.colab import userdata

# ========== PENGATURAN (ganti di sini saja) ==========
MODEL_NAME = "gemini-3.1-flash-lite"   # kalau kena 429 terus, coba: "gemini-3.5-flash" atau "gemini-3.1-flash-lite"
NOTA = ["/content/nota1.jpg", "/content/nota2.jpg"]

# ---------- Bentuk data ----------
class Item(BaseModel):
    name: str
    qty: int
    unit_price: float
    total_price: float

class Charge(BaseModel):
    name: str
    amount: float

class Receipt(BaseModel):
    items: list[Item]
    subtotal: float
    additional_charges: list[Charge]
    total: float

# ---------- Koneksi ke Gemini ----------
client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

PROMPT = """Baca struk ini. Ekstrak semua item (nama, jumlah, harga satuan, total harga item),
subtotal sebelum pajak/service, semua biaya tambahan (pajak, service charge, dll), dan total akhir.
Diskon ditulis sebagai angka negatif. Angka dalam Rupiah tanpa titik atau koma pemisah ribuan."""

if "CACHE" not in globals():
    CACHE = {}   # tempat menyimpan hasil supaya tidak memanggil Google berulang


def read_with_gemini(image_path, model_name=MODEL_NAME, retries=3):
    """Mengembalikan (hasil, waktu_asli_detik, nama_model)."""
    kunci = (image_path, model_name)
    if kunci in CACHE:
        return CACHE[kunci]

    image = Image.open(image_path)
    for attempt in range(retries):
        try:
            t0 = time.time()
            response = client.models.generate_content(
                model=model_name,
                contents=[image, PROMPT],
                config={"response_mime_type": "application/json",
                        "response_schema": Receipt},
            )
            hasil = (response.parsed, time.time() - t0, model_name)
            CACHE[kunci] = hasil
            return hasil
        except Exception as e:
            pesan = str(e)
            if "429" in pesan:
                print("Jatah habis (429). Pesan lengkap dari Google:\n")
                print(pesan)
                raise
            print(f"Percobaan {attempt + 1} gagal: {pesan[:80]}")
            time.sleep(10 * (attempt + 1))
    raise RuntimeError("Server sedang sibuk. Coba lagi beberapa menit lagi.")


# ---------- Kunci jawaban dari nota asli ----------
GROUND_TRUTH = {
    "/content/nota1.jpg": {"harga_item": [28000, 35000], "subtotal": 63000, "total": 69500},
    "/content/nota2.jpg": {"harga_item": [4000, 20300, 10200, 2100], "subtotal": 36600, "total": 32900},
}

def bandingkan(path, receipt):
    gt = GROUND_TRUTH[path]
    harga_ai = sorted(int(i.total_price) for i in receipt.items)
    print("  Harga item benar :", harga_ai == sorted(gt["harga_item"]))
    print("  Subtotal benar   :", int(receipt.subtotal) == gt["subtotal"])
    print("  Total benar      :", int(receipt.total) == gt["total"])

def cek_hitungan(receipt):
    jumlah_item = sum(i.total_price for i in receipt.items)
    biaya = sum(c.amount for c in receipt.additional_charges)
    print("  Jumlah item      :", jumlah_item, "| subtotal:", receipt.subtotal)
    print("  Subtotal + biaya :", receipt.subtotal + biaya, "| total:", receipt.total)


# ---------- Jalankan ----------
for path in NOTA:
    hasil, waktu, model_name = read_with_gemini(path)
    print("===", path, f"| model: {model_name} | waktu asli: {waktu:.1f} detik")
    print(hasil)
    bandingkan(path, hasil)
    cek_hitungan(hasil)
    print()

=== /content/nota1.jpg | model: gemini-3.1-flash-lite | waktu asli: 2.6 detik
items=[Item(name='Ice Matcha', qty=1, unit_price=28000.0, total_price=28000.0), Item(name='Ricebowl Ayam Teriyaki', qty=1, unit_price=35000.0, total_price=35000.0)] subtotal=63000.0 additional_charges=[Charge(name='Pajak', amount=6300.0), Charge(name='Pembulatan', amount=200.0)] total=69500.0
  Harga item benar : True
  Subtotal benar   : True
  Total benar      : True
  Jumlah item      : 63000.0 | subtotal: 63000.0
  Subtotal + biaya : 69500.0 | total: 69500.0

=== /content/nota2.jpg | model: gemini-3.1-flash-lite | waktu asli: 4.2 detik
items=[Item(name='Aqua Air Mineral Botol 600 ml', qty=1, unit_price=4000.0, total_price=4000.0), Item(name='Laurier Relax Night Pembalut Wanita Wing Malam 35 cm 12 pcs', qty=1, unit_price=20300.0, total_price=20300.0), Item(name='Roma Krim Crackers 107 g', qty=1, unit_price=10200.0, total_price=10200.0), Item(name='Alfamart Eco Paper Bag', qty=1, unit_price=2100.0, total_pr

Saya memilih Gemini (gemini-3.1-flash-lite). Pada 2 nota uji, Gemini membaca seluruh nama item, harga, subtotal, biaya tambahan, dan total dengan benar, dan hasilnya langsung sesuai skema data yang dibutuhkan aplikasi. Donut memang lebih cepat (sekitar 1,7 detik vs 2,6 sampai 4,2 detik) dan bisa berjalan offline, tetapi daftar itemnya berantakan: pada nota Alfamart, nama dan harga beberapa item hilang atau tertukar, dan alamat serta data lain ikut terbaca sebagai item. Karena split bill bergantung pada ketepatan item dan harga, akurasi lebih penting daripada selisih kecepatan beberapa detik.

# Step 1: Riset Model antara 2 Nota

**Tujuan:** membandingkan 2 model AI untuk membaca 2 nota (item, jumlah, harga, subtotal, biaya tambahan, total).

**Data uji:** 2 nota
- Nota 1: struk restoran (Warung Hagia), 2 item, ada pajak dan pembulatan
- Nota 2: struk pesanan Alfamart, 4 item, ada diskon per item dan biaya pengiriman

**Model yang dibandingkan:**
- **Model A: Donut** (`naver-clova-ix/donut-base-finetuned-cord-v2`), OCR-free, dijalankan lokal di Colab (GPU)
- **Model B: Gemini** (`gemini-3.1-flash-lite`), multimodal, dijalankan via API

---

## 1. Perbandingan Model

| Kriteria | Donut (cord-v2) | Gemini (3.1-flash-lite) |
|---|---|---|
| Waktu kecepatan nota 1 | 1,7 detik | 2,6 detik |
| Waktu kecepatan nota 2 | 1,8 detik | 4,2 detik |
| Nama item yg benar | Tidak (sering hilang atau tertukar) | Ya, semua benar |
| Harga item yg benar | Sebagian (nota 2: 1 dari 4 harga hilang, 1 nempel ke baris salah) | Ya, semua benar |
| Subtotal, biaya tambahan, total | Benar | Benar |
| Format keluaran | Tidak konsisten, banyak data tidak relevan (alamat, telepon, nama kasir) | Konsisten, langsung sesuai skema |
| Perlu kode tambahan | Banyak (memetakan hasil ke format aplikasi) | Tidak ada |
| Perlu internet / API key | Tidak | Ya |
| Perlu GPU | Disarankan | Tidak |
| Biaya dan batas pemakaian | Gratis, tanpa batas | Ada batas kuota, server bisa penuh (error 429 dan 503) |

## 2. Detail Hasil per Nota

| Bagian | Nota 1 Donut | Nota 1 Gemini | Nota 2 Donut | Nota 2 Gemini |
|---|---|---|---|---|
| Item terbaca benar | 2 dari 2 (satu nyangkut di struktur `sub`) | 2 dari 2 | 1 dari 4 lengkap, sisanya nama atau harga hilang | 4 dari 4 |
| Subtotal | Benar | Benar | Benar | Benar |
| Biaya tambahan | Benar (tanpa label jelas) | Benar | Diskon benar, pengiriman salah label | Benar |
| Total | Benar | Benar | Benar | Benar |
| Data tidak relevan ikut terbaca | Ya (nama toko, telepon, nama kasir) | Tidak | Ya (alamat, status order, email) | Tidak |

**Catatan pengukuran:** waktu Donut diukur di GPU Colab dan tidak menghitung waktu memuat model. Waktu Gemini mencakup perjalanan lewat web gemini.

## 3. Analisis

- **Akurasi:** Gemini membaca seluruh item, harga, subtotal, biaya tambahan, dan total dengan benar di kedua nota. Donut benar pada bagian subtotal dan total, tetapi daftar item berantakan terutama di nota 2 (nama panjang dua baris, banyak teks di sekitar item).

- **Kecepatan:** Donut sedikit lebih cepat (sekitar 1,7 detik vs 2,6 sampai 4,2 detik), tetapi selisihnya hanya beberapa detik sehingga tidak sebanding dengan selisih akurasi.

- **Kemudahan integrasi:** Gemini bisa dipaksa mengembalikan JSON sesuai skema, sehingga hasilnya langsung dipakai aplikasi. Hasil Donut harus dibersihkan dan dipetakan dulu.

- **Ketersediaan:** Gemini bergantung pada internet dan kuota (sempat mengalami error 503 server penuh dan 429 kuota habis). Donut bisa jalan offline dan tanpa batas.
- **Validasi otomatis:** jumlah harga item = subtotal dan subtotal + biaya tambahan = total, terpenuhi pada semua hasil Gemini.

## 4. Model Terpilih: Gemini (`gemini-3.1-flash-lite`)

Pada 2 nota uji, Gemini membaca semua nama item, harga, subtotal, biaya tambahan, dan total dengan benar, dan hasilnya langsung sesuai skema data yang dibutuhkan aplikasi. Donut memang lebih cepat dan bisa berjalan offline, tetapi daftar itemnya berantakan. Sehingga nota Alfamart, nama dan harga beberapa item hilang atau tertukar, dan alamat serta data lain ikut terbaca sebagai item. Karena fitur split bill bergantung pada ketepatan item dan harga, akurasi lebih penting daripada selisih kecepatan beberapa detik.

## 5. Kelemahan yang Ditemukan

• Gemini rawan tumbang: Karena basisnya API, sistem kita sangat bergantung pada koneksi internet. Kalau server Google lagi kepenuhan atau kuota habis, aplikasi langsung mogok dan muncul eror 503 atau 429.

• Informasi diskon jebol: Di nota Alfamart, potongan harga per item (seperti diskon Aqua atau Roma) malah digabung jadi satu baris "Total Diskon" di bawah. Efeknya, kita jadi kehilangan jejak diskon itu sebenarnya milik barang yang mana.

• Donut kurang jeli: Model Donut masih kewalahan kalau harus membaca tulisan yang ukurannya kekecilan atau terpotong. Selain itu, dia gampang bingung dan ikut membaca teks lain di luar daftar belanjaan.

• Data uji masih minim: Sejauh ini kita baru mengetes pakai 2 contoh nota saja. Jadi, kita belum tahu pasti bagaimana performanya kalau dihadapkan dengan nota yang kameranya miring, fotonya buram, atau dari toko lain yang formatnya beda.

## 6. Ide Perbaikan

• Siasati diskon sejak awal: Kita bakal tweak perintahnya agar AI langsung memotong harga barang di tempat jika ada diskon per item. Jadi data harga yang masuk sudah bersih.

• Pasang sistem cadangan: Biar aplikasi tidak gampang mogok, kita perlu buat mekanisme caching dan menyiapkan skenario failover. Kalau API Gemini lagi sibuk, sistem bisa otomatis dialihkan.

• Siapkan model offline: Kita bisa pakai Donut atau model lokal sejenis sebagai backup utama saat tidak ada internet, jadi aplikasi tetap bisa jalan secara offline.

• Andalkan validasi mandiri: Kita optimalkan fitur validasi otomatis dan tabel edit yang sudah ada di aplikasi. Jadi kalau AI salah baca angka, pengguna bisa langsung memperbaikinya sendiri dengan mudah.

• Uji coba lebih brutal: Kita harus kumpulkan lebih banyak variasi struk belanjaan, mulai dari yang kusut sampai yang pudar, biar bisa mengukur seberapa tangguh sistem ini secara riil.